# Zotero cleaner

In [ ]:
import os
import shutil
from pyzotero import zotero

from dotenv import load_dotenv
import os

from pathlib import Path

# Load environment variables from .env file
load_dotenv()

In [ ]:
# Configuration
LIBRARY_ID = os.getenv("LIBRARY_ID")  # Zotero library ID
LIBRARY_TYPE = os.getenv("LIBRARY_TYPE", "user")  # Use 'group' for group libraries
ZOTERO_KEY = os.getenv("ZOTERO_KEY")  # API key

ZOTERO_STORAGE_PATH = Path(os.getenv("ZOTERO_STORAGE_PATH", "zotero_storage"))  # Path to store attachments
DEST_FOLDER = Path(os.getenv("DEST_FOLDER", "zotero_export"))  # Path to store exported files

In [ ]:
# Initialize Zotero API
zot = zotero.Zotero(LIBRARY_ID, LIBRARY_TYPE, ZOTERO_KEY)

Get all items in the Zotero library

In [ ]:
all_items = []
start = 0
limit = 100  # Number of items to fetch per request

while True:
    items = zot.items(start=start, limit=limit)  # Fetch a batch of items
    if not items:  # Stop if no more items are returned
        break
    all_items.extend(items)  # Add items to the list
    start += limit  # Update start for the next batch

print(f"Total items retrieved: {len(all_items)}")

Move local files to specified new storage and delete the original file

In [ ]:
for item in all_items:

    # Check if the attachment is a PDF and stored locally
    if item['data'].get('linkMode') != 'imported_file' or not item['data']['filename'].endswith('.pdf'):
        continue

    # Get attachment details
    key = item['key']
    parent_key = item['data'].get('parentItem')  # Use .get() to handle missing parentItem
    title = item['data']['title']
    file_name = item['data']['filename']
    storage_path = os.path.join(ZOTERO_STORAGE_PATH, key)

    # Skip if the storage folder does not exist
    if not os.path.exists(storage_path):
        print(f"Storage path not found for attachment key {key}")
        continue

    # Identify the file to move
    files = [f for f in os.listdir(storage_path) if os.path.isfile(os.path.join(storage_path, f))]
    
    # Filter to get the PDF file
    try:
        pdf_file = [f for f in files if f.endswith('.pdf')][0]
    except Exception as e:
        print('Error finding ', files)
        continue
    
    if not files:
        print(f"No files found in {storage_path}")
        continue

    file_path = os.path.join(storage_path, pdf_file)
    dest_path = os.path.join(DEST_FOLDER, key, pdf_file)
    
    
    # Check if there is a .zotero-reader-state file and move it as well
    reader_state_file = [f for f in files if f == '.zotero-reader-state']

    if reader_state_file:
        reader_state_file_path = os.path.join(storage_path, reader_state_file[0])
        reader_state_dest_path = os.path.join(DEST_FOLDER, key, reader_state_file[0])
        try:
            if not os.path.exists(os.path.dirname(reader_state_dest_path)):
                os.makedirs(os.path.dirname(reader_state_dest_path))
            shutil.move(reader_state_file_path, reader_state_dest_path)
            print(f"Moved {reader_state_file[0]} to {reader_state_dest_path}")
        except Exception as e:
            print(f"Error moving {reader_state_file[0]}: {e}")

    # Move the file to OneDrive folder
    try:
        if not os.path.exists(os.path.dirname(dest_path)):
            os.makedirs(os.path.dirname(dest_path))
        shutil.move(file_path, dest_path)
        print(f"Moved {pdf_file} to {dest_path}")
    except Exception as e:
        print(f"Error moving {pdf_file}: {e}")
        continue

    # Create the linked file path
    linked_file_path = os.path.abspath(dest_path)

    # Create a new linked file attachment
    try:
        new_attachment = {
            'itemType': 'attachment',
            'parentItem': parent_key,  # Can be None for standalone attachments
            'linkMode': 'linked_file',
            'title': title,
            'path': linked_file_path,  # Path to the linked file
            'contentType': 'application/pdf',  # Specify the content type for PDFs
            'note': 'Moved to OneDrive folder',
        }
        zot.create_items([new_attachment])
        print(f"Created new linked file attachment for {pdf_file} at {linked_file_path}")
    except Exception as e:
        print(f"Error creating linked file attachment for {pdf_file}: {e}")
        continue

    # Delete the original Zotero item
    try:
        zot.delete_item(item)
        print(f"Deleted original attachment with key {key}")
    except Exception as e:
        print(f"Error deleting Zotero item {key}: {e}")